# 2 · Extractor de cifras y verificador

El verificador extrae cada número del texto, deduce de qué indicador, país y año habla (o usa la referencia estructurada), y lo compara con la capa de KPIs con tolerancia de redondeo y compatibilidad de unidades.

In [1]:
import sys, os
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
from kpis.numeros import extraer
from kpis.verificador import verificar_texto

## Casos difíciles de formato

In [2]:
casos = ['bajó −1,9 %', 'una contracción del 3.5%', 'subió 2,6 puntos porcentuales',
         '6.827 USD', '6,718.14 dólares', '123,8 mil millones de USD', '121.000 millones de dólares',
         '18,1 millones de habitantes', 'US$ 1,2 billones', 'el tercer lugar en remesas', 'puesto 18 de 18',
         'En 2024 no hubo cambios']
pd.DataFrame([{'texto': s, 'cifras': [(c.texto, c.candidatos, c.unidad, c.decimales) for c in extraer(s)]}
              for s in casos])

,texto,cifras
0,"bajó −1,9 %","[(1,9, [-1.9], %, 1)]"
1,una contracción del 3.5%,"[(3.5, [3.5], %, 1)]"
2,"subió 2,6 puntos porcentuales","[(2,6, [2.6], pp, 1)]"
3,6.827 USD,"[(6.827, [6827.0, 6.827], USD, 0)]"
4,"6,718.14 dólares","[(6,718.14, [6718.14], USD, 2)]"
5,"123,8 mil millones de USD","[(123,8 mil millones, [123800000000.0], USD, 1)]"
6,121.000 millones de dólares,"[(121.000 millones, [121000000000.0, 121000000..."
7,"18,1 millones de habitantes","[(18,1 millones, [18100000.0], personas, 1)]"
8,"US$ 1,2 billones","[(1,2 billones, [1200000000000.0, 1200000000.0..."
9,el tercer lugar en remesas,"[(tercer lugar, [3.0], puesto, 0)]"


Cuando el texto es ambiguo (`6,718`, `billones`), el extractor devuelve **varios candidatos** y el verificador acepta el que coincida. Los años sueltos no son cifras.

## Verificación en contexto

In [3]:
frases = [
  'En 2024 la economía de Ecuador se contrajo 1,9 %, frente a una mediana regional de 3,4 %.',
  'La inflación fue de 2,2 % en 2024.',
  'La inflación bajó a 1,5 %, 0,7 puntos porcentuales menos que en 2023.',
  'Las remesas crecieron 0,8 % en 2024.',
  'La IED fue de 0,4 % del PIB, el tercer puesto más bajo.',
  'Se espera un repunte de 4,2 % el próximo año.',
  'El PIB creció 2,5 % en 2024.']
filas = []
for f in frases:
    for c in verificar_texto(f, 'ECU', 2024):
        filas.append({'frase': f[:60], 'cifra': c.texto, 'estado': c.estado, 'real': c.real_formateado,
                      'hecho': c.hecho_id, 'marcas': c.marcas})
pd.DataFrame(filas)

,frase,cifra,estado,real,hecho,marcas
0,"En 2024 la economía de Ecuador se contrajo 1,9...","1,9",correcta,"-1,9 %",crecimiento|ECU|2024|valor,[]
1,"En 2024 la economía de Ecuador se contrajo 1,9...","3,4",correcta,"3,4 %",crecimiento|LAT|2024|mediana_region,[]
2,"La inflación fue de 2,2 % en 2024.","2,2",incorrecta,"1,5 %",inflacion|ECU|2024|valor,[]
3,"La inflación bajó a 1,5 %, 0,7 puntos porcentu...","1,5",correcta,"1,5 %",inflacion|ECU|2024|valor,[]
4,"La inflación bajó a 1,5 %, 0,7 puntos porcentu...","0,7",correcta,"-0,7 pp",inflacion|ECU|2024|variacion,[]
5,"Las remesas crecieron 0,8 % en 2024.","0,8",correcta,"0,8 pp",remesas|ECU|2024|variacion,[pp_escrito_como_%]
6,"La IED fue de 0,4 % del PIB, el tercer puesto ...","0,4",correcta,"0,4 %",ied|ECU|2024|valor,[]
7,"La IED fue de 0,4 % del PIB, el tercer puesto ...",tercer puesto,correcta,puesto 16 de 18,ied|ECU|2024|ranking,[ordinal]
8,"Se espera un repunte de 4,2 % el próximo año.","4,2",sin_fuente,NaN,NaN,[]
9,"El PIB creció 2,5 % en 2024.","2,5",correcta,"2,5 %",pib_usd|ECU|2024|variacion,[indicador_secundario]


Dos casos para leer con calma:

- «Las remesas crecieron 0,8 %» es **correcta pero mal expresada**: el cambio real es +0,8 pp. Se acepta y se marca `pp_escrito_como_%`.
- «El PIB creció 2,5 % en 2024» se acepta porque el **PIB en dólares corrientes** subió 2,5 %, pero el PIB real cayó 1,9 %. El número existe; la frase engaña. Esto es una limitación de cualquier verificador de cifras: comprueba números, no la lectura que hace un gerente.